# Task 2: Profile and Clean H2H Data

In [2]:
import os
import json
import glob
import pandas as pd

RAW_DIR = "../data/raw"

# Find every H2H raw file fetched so far (works whether it's 90 or 153 files)
h2h_files = glob.glob(f"{RAW_DIR}/h2h_*.json")
h2h_files = [f for f in h2h_files if "progress" not in f]  # skip the progress tracker file

print(f"Found {len(h2h_files)} H2H raw files")

# Load and combine every fixture from every file into one flat list
all_fixtures = []
for filepath in h2h_files:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    all_fixtures.extend(data.get("response", []))

print(f"Total fixture records before dedup: {len(all_fixtures)}")

# Flatten the nested JSON into a table
df = pd.json_normalize(all_fixtures)

# Two team pairs can both include the same match once each is combined
# (e.g. h2h_2932_2938.json and a broader team's file might both contain
# the same Hilal-Ittihad fixture) - drop duplicates by fixture.id, the
# stable unique identifier from the API.
df = df.drop_duplicates(subset="fixture.id")

print(f"Total unique fixtures after dedup: {len(df)}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
print()
print(df.head())

# Basic profiling: dtype, null count, percent null, unique count, sample value
profile_rows = []
for col in df.columns:
    profile_rows.append({
        "column": col,
        "dtype": str(df[col].dtype),
        "null_count": df[col].isnull().sum(),
        "percent_null": round(df[col].isnull().mean() * 100, 1),
        "unique_count": df[col].nunique(),
        "sample_value": df[col].dropna().iloc[0] if df[col].notna().any() else None,
    })

profile_df = pd.DataFrame(profile_rows)
print("\nProfiling summary:")
print(profile_df.to_string(index=False))

Found 153 H2H raw files
Total fixture records before dedup: 1871
Total unique fixtures after dedup: 1871
Columns (40): ['fixture.id', 'fixture.referee', 'fixture.timezone', 'fixture.date', 'fixture.timestamp', 'fixture.periods.first', 'fixture.periods.second', 'fixture.venue.id', 'fixture.venue.name', 'fixture.venue.city', 'fixture.status.long', 'fixture.status.short', 'fixture.status.elapsed', 'fixture.status.extra', 'league.id', 'league.name', 'league.country', 'league.logo', 'league.flag', 'league.season', 'league.round', 'league.standings', 'teams.home.id', 'teams.home.name', 'teams.home.logo', 'teams.home.winner', 'teams.away.id', 'teams.away.name', 'teams.away.logo', 'teams.away.winner', 'goals.home', 'goals.away', 'score.halftime.home', 'score.halftime.away', 'score.fulltime.home', 'score.fulltime.away', 'score.extratime.home', 'score.extratime.away', 'score.penalty.home', 'score.penalty.away']

   fixture.id fixture.referee fixture.timezone               fixture.date  \
0      